# Random Forest — Modelo Global (PreTec21 → Tec21)
## Predicción de Deserción Estudiantil · Equipo 7

---
**Prerrequisito:** `clustering/02_kmeans_independiente.ipynb`

Entrena un **Random Forest** con búsqueda de hiperparámetros (RandomizedSearchCV),
validación cruzada 5-fold, y evalúa en Tec21 (cross-regime).
Muestra matriz de confusión, curvas ROC/PR y métricas completas.


## Setup e Importaciones

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pickle
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection  import (StratifiedKFold, cross_val_score,
                                       RandomizedSearchCV, train_test_split)
from sklearn.metrics          import (roc_auc_score, recall_score, f1_score,
                                       precision_score, roc_curve, auc,
                                       precision_recall_curve, average_precision_score,
                                       ConfusionMatrixDisplay, confusion_matrix)
from sklearn.impute           import SimpleImputer
from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print('✓ XGBoost:', xgb.__version__)
except ImportError:
    XGB_AVAILABLE = False
    print('⚠ XGBoost no disponible')

SEED = 42
np.random.seed(SEED)
print('✓ Librerías cargadas')

## Configuración y carga de datos

In [ ]:
PROCESSED_DIR = Path('../../../data/processed')
IMG_DIR       = Path('images'); IMG_DIR.mkdir(exist_ok=True)

TARGET    = 'retention'
MIN_AUC   = 0.60

df_pre = pd.read_csv(PROCESSED_DIR / 'df_pre.csv')
df_tec = pd.read_csv(PROCESSED_DIR / 'df_tec.csv')
print(f'✓ df_pre: {df_pre.shape}  |  df_tec: {df_tec.shape}')

## §8 — Features y Split

In [ ]:
FEATURE_COLS_CANDIDATES = [
    'PNA', 'admission_test_norm', 'english.evaluation', 'admission.rubric',
    'general.math.eval', 'online.test', 'FTE', 'apoyo_financiero',
    'has_extracurriculars', 'has_physical', 'has_cultural', 'has_social',
    'first_gen_enc', 'educ_padres_max', 'socioec_enc', 'social_lag_enc',
    'age', 'is_male', 'estuvo_prepa_tec',
    'foreign_Yes: Foreigner', 'foreign_Yes: National',
    'school_enc', 'region_enc',
    'first_gen_present', 'parents_edu_present', 'took_admission_test',
    'has_socioeconomic_data', 'has_social_lag_data', 'has_zone_data',
]
EXCLUDE = {TARGET, 'cluster', 'generation', 'regime', 'educational.model'}
seen = set()
FEATURE_COLS = [c for c in FEATURE_COLS_CANDIDATES
                if c in df_pre.columns and c not in EXCLUDE
                and c not in seen and not seen.add(c)]
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

def make_split(df_regime, feature_cols, target=TARGET, seed=SEED):
    cols = [c for c in feature_cols if c in df_regime.columns]
    X = df_regime[cols].values.astype(float)
    y = df_regime[target].values.astype(int)
    X = SimpleImputer(strategy='median').fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    return X_tr, X_te, y_tr, y_te, cols

X_tr_pre, X_te_pre, y_tr_pre, y_te_pre, feat_pre = make_split(df_pre, FEATURE_COLS)
X_tr_tec, X_te_tec, y_tr_tec, y_te_tec, feat_tec = make_split(df_tec, FEATURE_COLS)
print(f'PreTec21  train={X_tr_pre.shape}  test={X_te_pre.shape}')
print(f'Tec21     train={X_tr_tec.shape}  test={X_te_tec.shape}')

## Función de evaluación

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name, feat_cols,
                   threshold=None, n_bootstrap=200, seed=SEED):
    model.fit(X_tr, y_tr)
    has_proba = hasattr(model, 'predict_proba')
    y_proba   = model.predict_proba(X_te)[:,1] if has_proba else None
    if threshold is None and has_proba:
        skf_tmp = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
        oof = np.zeros(len(X_tr))
        for tr_i, va_i in skf_tmp.split(X_tr, y_tr):
            model.fit(X_tr[tr_i], y_tr[tr_i])
            oof[va_i] = model.predict_proba(X_tr[va_i])[:,1]
        model.fit(X_tr, y_tr)
        oof_dropout = 1.0 - oof
        prec_oof, rec_oof, thr_oof = precision_recall_curve(y_tr, oof_dropout, pos_label=0)
        f1_oof  = 2*prec_oof*rec_oof/(prec_oof+rec_oof+1e-8)
        thr_best = float(thr_oof[np.argmax(f1_oof[:-1])])
        threshold = 1.0 - thr_best
    y_pred = (y_proba >= threshold).astype(int) if (threshold is not None and has_proba)              else model.predict(X_te)
    auc_val  = roc_auc_score(y_te, y_proba) if has_proba else 0.5
    rec  = recall_score(y_te, y_pred, zero_division=0)
    prec = precision_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, pos_label=0, zero_division=0)
    rng  = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(y_te), len(y_te))
        if y_proba is not None and len(np.unique(y_te[idx])) > 1:
            aucs.append(roc_auc_score(y_te[idx], y_proba[idx]))
    ci_lo, ci_hi = np.percentile(aucs,[2.5,97.5]) if aucs else (auc_val, auc_val)
    print(f'  {model_name:<28} AUC={auc_val:.3f} [{ci_lo:.3f},{ci_hi:.3f}]'
          f'  Recall={rec:.3f}  F1={f1:.3f}  thr={threshold:.2f}')
    return dict(model=model_name, auc=auc_val, ci_lo=ci_lo, ci_hi=ci_hi,
                recall=rec, precision=prec, f1=f1, threshold=threshold,
                y_proba=y_proba, y_pred=y_pred, feat_cols=feat_cols)

## §9a — Búsqueda de Hiperparámetros (RandomizedSearchCV)

Espacio RF: `n_estimators∈{100,200,300,400}`, `max_depth∈{4,6,8,10}`,
`min_samples_leaf∈{5,10,20}`, `max_features∈{'sqrt','log2'}`.
Métrica: AUC-ROC, 5-fold StratifiedKFold.

In [ ]:
rf_param_dist = {
    'n_estimators'    : [100, 200, 300, 400],
    'max_depth'       : [4, 6, 8, 10],
    'min_samples_leaf': [5, 10, 20],
    'max_features'    : ['sqrt', 'log2'],
}
rf_base   = RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1)
rf_search = RandomizedSearchCV(
    rf_base, rf_param_dist, n_iter=30, scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    random_state=SEED, n_jobs=-1, verbose=0
)
rf_search.fit(X_tr_pre, y_tr_pre)
best_rf_params = rf_search.best_params_
print('✓ Mejores parámetros RF:')
for k, v in sorted(best_rf_params.items()):
    print(f'    {k}: {v}')
print(f'  CV AUC (media): {rf_search.best_score_:.4f}')

## §9b — Validación Cruzada 5-fold

In [ ]:
from sklearn.model_selection import cross_val_score, learning_curve

cv_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

scores_rf = cross_val_score(
    RandomForestClassifier(**best_rf_params, class_weight='balanced',
                            random_state=SEED, n_jobs=-1),
    X_tr_pre, y_tr_pre, cv=cv_skf, scoring='roc_auc', n_jobs=-1
)
print(f'5-fold CV AUC (RF, PreTec21): {scores_rf.mean():.4f} ± {scores_rf.std():.4f}')
cv_results = {'RandomForest': scores_rf}

rf_lc = RandomForestClassifier(**best_rf_params, class_weight='balanced',
                                random_state=SEED, n_jobs=-1)
train_sizes, train_scores, val_scores = learning_curve(
    rf_lc, X_tr_pre, y_tr_pre,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=cv_skf, scoring='roc_auc', n_jobs=-1
)
lc_train_mean = train_scores.mean(axis=1); lc_train_std  = train_scores.std(axis=1)
lc_val_mean   = val_scores.mean(axis=1);  lc_val_std    = val_scores.std(axis=1)

print('\n═══ Learning Curve (RF) ═══')
print(f'  {"Tamaño":>8}  {"Train":>8}  {"Val":>8}  {"Gap":>7}')
for sz, tr, va in zip(train_sizes, lc_train_mean, lc_val_mean):
    print(f'  {int(sz):>8d}  {tr:>8.4f}  {va:>8.4f}  {tr-va:>7.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.fill_between(train_sizes, lc_train_mean-lc_train_std, lc_train_mean+lc_train_std,
                alpha=0.15, color='steelblue')
ax.fill_between(train_sizes, lc_val_mean-lc_val_std, lc_val_mean+lc_val_std,
                alpha=0.15, color='darkorange')
ax.plot(train_sizes, lc_train_mean, 'o-', color='steelblue', label='Train AUC')
ax.plot(train_sizes, lc_val_mean,   'o-', color='darkorange', label='Val AUC (CV)')
ax.set_xlabel('Tamaño entrenamiento'); ax.set_ylabel('AUC-ROC')
ax.set_title('Curva de Aprendizaje — Random Forest')
ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.bar(['Random Forest'], [scores_rf.mean()], yerr=[scores_rf.std()],
        capsize=5, color='#55A868', alpha=0.8)
ax2.set_ylim(0.5, 0.75); ax2.set_ylabel('AUC-ROC (5-fold CV)')
ax2.set_title('AUC-ROC en CV — Random Forest')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(IMG_DIR / 'rf_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figura guardada')

## §9c — Entrenamiento final y evaluación en Tec21

In [ ]:
rf_final = RandomForestClassifier(**best_rf_params, class_weight='balanced',
                                    random_state=SEED, n_jobs=-1)
print('Evaluando Random Forest (PreTec21 → Tec21):')
res_rf = evaluate_model(rf_final, X_tr_pre, X_te_tec, y_tr_pre, y_te_tec,
                        'RandomForest', feat_pre)

## Matriz de Confusión y Curvas ROC/PR

In [ ]:
# ── Matriz de Confusión ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_te_tec, res_rf['y_pred'], normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Desertor','Retenido'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues', values_format='.2f')
axes[0].set_title(f'Matriz de Confusión (norm.)\nAUC={res_rf["auc"]:.3f}')

# ── Curvas ROC y PR ───────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_te_tec, res_rf['y_proba'])
roc_auc_val  = auc(fpr, tpr)
prec_c, rec_c, _ = precision_recall_curve(y_te_tec, res_rf['y_proba'])
ap_val = average_precision_score(y_te_tec, res_rf['y_proba'])

axes[1].plot(fpr, tpr, color='#4C72B0', lw=2, label=f'ROC (AUC={roc_auc_val:.3f})')
axes[1].plot([0,1],[0,1],'k--', lw=1, alpha=0.5)
ax2 = axes[1].twinx()
ax2.plot(rec_c, prec_c, color='#DD8452', lw=2, linestyle='--', label=f'PR (AP={ap_val:.3f})')
axes[1].set_xlabel('FPR / Recall'); axes[1].set_ylabel('TPR', color='#4C72B0')
ax2.set_ylabel('Precisión', color='#DD8452')
axes[1].set_title('Curvas ROC y Precision-Recall (Tec21)')
lines1, labs1 = axes[1].get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1+lines2, labs1+labs2, loc='lower left', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(IMG_DIR / 'rf_metricas.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Guardado: {IMG_DIR}/rf_metricas.png')

# ── Métricas resumen ──────────────────────────────────────────────────────────
print('\n═══ Métricas Finales — RF ═══')
print(f'  AUC-ROC  : {res_rf["auc"]:.4f} [{res_rf["ci_lo"]:.3f}, {res_rf["ci_hi"]:.3f}]')
print(f'  Recall   : {res_rf["recall"]:.4f}')
print(f'  Precision: {res_rf["precision"]:.4f}')
print(f'  F1 (0)   : {res_rf["f1"]:.4f}')
print(f'  Umbral   : {res_rf["threshold"]:.4f}')

## Importancia de Variables

In [ ]:
rf_final.fit(X_tr_pre, y_tr_pre)
fi = pd.Series(rf_final.feature_importances_, index=feat_pre).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
fi.plot(kind='barh', ax=ax, color='#2563eb', alpha=0.8)
ax.set_title('Random Forest — Top 15 Variables (Importancia Gini)')
ax.set_xlabel('Importancia'); ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / 'rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Guardado: {IMG_DIR}/rf_feature_importance.png')

## Guardar modelo

In [ ]:
import json

# ── Modelo RF → pickle ────────────────────────────────────────────────────────
with open(PROCESSED_DIR / 'rf_model.pkl', 'wb') as f:
    pickle.dump(rf_final, f)
print('✓ rf_model.pkl guardado')

# ── Arrays de test → npy ─────────────────────────────────────────────────────
np.save(PROCESSED_DIR / 'X_te_pre.npy', X_te_pre)
np.save(PROCESSED_DIR / 'X_te_tec.npy', X_te_tec)
np.save(PROCESSED_DIR / 'y_te_pre.npy', y_te_pre)
np.save(PROCESSED_DIR / 'y_te_tec.npy', y_te_tec)
print('✓ Arrays de test guardados')

# ── Lista de features → JSON ──────────────────────────────────────────────────
with open(PROCESSED_DIR / 'feat_pre.json', 'w') as f:
    json.dump(feat_pre, f)
print('✓ feat_pre.json guardado')
print('→ Siguiente: modelos/shap_rf_clusters.ipynb')